In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (Hemolytic-Pred)
This notebook processes the **Hemolytic-Pred** source, which provides peptide sequences without explicit hemolytic or non-hemolytic labels. The goal is to extract, standardize, and deduplicate all available sequences, and to store them as an **unlabeled dataset** for potential downstream use (e.g., pretraining, external validation, or exploratory analysis).

- **Toxic effect / endpoint:** hemolytic
- **Source:** Hemolytic-Pred
- **Label status**: sequences are treated as unlabeled, as the dataset does not provide explicit hemolytic annotations.
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:


- **Loads all FASTA-like files** (`.fasta`, `.fa`, `.faa`, `.txt`) found in the Hemolytic-Pred input directory.
- **Parses peptide sequences** into a unified table.
- **Assigns a placeholder label (`label = 2`)** to indicate that these sequences are **unlabeled**:
  - `0` → non-hemolytic (used in other datasets),
  - `1` → hemolytic (used in other datasets),
  - `2` → unknown / unlabeled (this source).
- **Checks duplicated sequences**:
  - unique sequences are retained,
  - duplicates with consistent placeholder labels are collapsed,
  - any unexpected inconsistencies are flagged as errors.
- **Builds metadata** from the project-wide Excel description sheet and appends QC statistics.
- **Exports outputs**:
  - `detected_unlabel_sequences.csv`
  - `metadata.json`.

In [2]:
name_source = "Hemolytic-Pred"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
input_dir = Path(PATH_INPUT) / name_source
valid_ext = {".fasta", ".fa", ".faa", ".txt"}
dfs = []

for file in input_dir.iterdir():
    if file.is_file() and file.suffix.lower() in valid_ext:
        df = read_fasta_doc(file)
        df["source_file"] = file.name
        dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

In [4]:
df_unlabel = (df
      .assign(label=2)
      [["sequence", "label"]]
) # There is no information about the labels of this source, therefore it will be identified with a 2

- Checking duplicates

In [5]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df_unlabel, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [6]:
df_full.shape

(1235, 2)

In [7]:
df_errors.shape

(0, 1)

- Working with metada

In [8]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [9]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_unlabel_sequences": int((df_full["label"] == 2).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2023,
 'last update date': datetime.datetime(2023, 7, 5, 0, 0),
 'download date': Timestamp('2024-08-01 00:00:00'),
 'file format': 'fasta',
 'peptide property': 'hemolytic, toxic',
 'dataset information': 'No information',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'No information',
 'repository or server': 'Supplementary material of the paper',
 'publication': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10331097/',
 'number_of_raw_sequences': 1235,
 'number_of_sequences_retained': 1235,
 'number_of_positive_sequences': 0,
 'number_of_negative_sequences': 0,
 'number_of_unlabel_sequences': 1235,
 'number_of_erroneous_sequences': 0,
 'modified_sequences_included': False}

- Exporting data

In [10]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [11]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_unlabel_sequences.csv", index=False)